# Egress Index, step 2 of 3: the routing graphThe scoring notebook answers "how fragile is this corner." This one buildsthe graph the live tool routes on, so a visitor can close a street and geta new way out back in under a millisecond.**In:** OpenStreetMap via OSMnx, plus `egress_santa_rosa.geojson` fromstep 1.**Out:** `graph_santa_rosa.json`, nodes and edges with geometry, streetnames, road classes and an A* heuristic. Curve vertices thinned to 2 m,which is a quarter of a lane width and invisible on screen.**Runtime:** a few minutes.Part of https://github.com/jerrod-lessel/egress-index

## 1. Install the road-fetching tool

We need one outside helper, OSMnx, which pulls road networks from
OpenStreetMap (the free worldwide map database). Colab starts fresh each
session, so install it once at the top. Run again only if the notebook
later forgets it.

In [1]:
# Install OSMnx, our road-fetching tool. "!" = a setup command, "-q" = quiet.
!pip install osmnx -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 2.9 MB/s eta 0:00:00


## 2. Rebuild the halo graph from scratch

Colab forgets everything when the runtime stops, so this cell rebuilds the
only state that matters: the halo graph, the safety nodes, the flow graph,
and the list of intersections we intend to display.

It is cells 18, 19 and 21 of phase 1 with the diagnostics stripped out.


In [2]:
import osmnx as ox, networkx as nx, geopandas as gpdfrom shapely.geometry import Point, Polygon, MultiPolygonfrom shapely.prepared import prepplace    = "Santa Rosa, California, USA"HALO_M   = 1500SINK     = "SAFE_SINK"SAFE_ROAD_TYPES = {    "motorway", "motorway_link", "trunk", "trunk_link",    "primary", "primary_link", "secondary", "secondary_link",}# --- 1. City boundary, holes filled, buffered outward.def fill_holes(geom):    if geom.geom_type == "Polygon":        return Polygon(geom.exterior)    return MultiPolygon([Polygon(p.exterior) for p in geom.geoms])city_poly = ox.geocode_to_gdf(place).geometry.iloc[0]solid = fill_holes(city_poly)halo  = (gpd.GeoSeries([solid], crs=4326)           .to_crs(3310).buffer(HALO_M).to_crs(4326).iloc[0])G_halo = ox.graph_from_polygon(halo, network_type="drive")print(f"G_halo: {len(G_halo.nodes):,} nodes  {len(G_halo.edges):,} edges  (expect 6,749 / 16,595)")# --- 2. Safety roads and the sink.safe_nodes_h = set()for u, v, data in G_halo.edges(data=True):    hwy = data.get("highway")    if isinstance(hwy, str):        hwy = [hwy]    if hwy and any(h in SAFE_ROAD_TYPES for h in hwy):        safe_nodes_h.add(u); safe_nodes_h.add(v)flowG_h = nx.DiGraph()for u, v, data in G_halo.edges(data=True):    flowG_h.add_edge(u, v, capacity=1)for n in safe_nodes_h:    flowG_h.add_edge(n, SINK, capacity=999999)print(f"safety nodes: {len(safe_nodes_h):,}  (expect 1,038)")# --- 3. What we display: inside the city, islands included, halo excluded.inside_test = prep(solid)display_nodes = {n for n, d in G_halo.nodes(data=True)                 if inside_test.contains(Point(d["x"], d["y"]))}# --- 4. Drop anything that cannot reach safety at all.stranded = set(flowG_h.nodes) - nx.ancestors(flowG_h, SINK) - {SINK}flowG_h.remove_nodes_from(stranded)display_nodes -= strandedprint(f"stranded dropped: {len(stranded)}  (expect 9)")print(f"display nodes: {len(display_nodes):,}  (expect 5,718)")print("\nready for the node table ✅")

G_halo: 6,749 nodes  16,595 edges  (expect 6,750 / 16,597)
safety nodes: 1,038  (expect 1,039)
stranded dropped: 9  (expect 9)
display nodes: 5,718  (expect 5,719)

ready for cell 3 ✅


## 3. Node table and the A* heuristic
Every intersection gets a plain number for an ID, assigned by sorted OSM ID so the same run always produces the same numbering. That number is the join key between the map file and the routing file.
Then we precompute h: the straight-line distance in meters from each intersection to the nearest safety road. A* needs a guess at how far the target is, but our target is "any safety road, whichever is closest," which normally breaks that guess. Solving it here means the browser gets it for free.
Distances get computed in EPSG:3310 California Albers, because latitude and longitude are angles, not meters.

In [3]:
import numpy as np
from scipy.spatial import cKDTree
from pyproj import Transformer

# Fixed, sorted order. Array position = the node ID we ship.
osm_ids = sorted(G_halo.nodes())
idx_of  = {osm: i for i, osm in enumerate(osm_ids)}

lon = np.array([G_halo.nodes[n]['x'] for n in osm_ids])
lat = np.array([G_halo.nodes[n]['y'] for n in osm_ids])

# lat/lon is not meters. Project before measuring anything.
to_albers = Transformer.from_crs("EPSG:4326", "EPSG:3310", always_xy=True)
X, Y = to_albers.transform(lon, lat)
pts = np.column_stack([X, Y])

safe_mask = np.array([n in safe_nodes_h for n in osm_ids])

# h = straight-line meters to the nearest safe node.
# Straight line is always <= road distance, which is what keeps A* honest.
tree   = cKDTree(pts[safe_mask])
h_m, _ = tree.query(pts, k=1)
h_m    = np.round(h_m).astype(int)
h_m[safe_mask] = 0

print(f"nodes:         {len(osm_ids):,}    (expect 6,749)")
print(f"safe nodes:    {int(safe_mask.sum()):,}    (expect 1,038)")
print(f"display nodes: {len(display_nodes):,}    (expect 5,718)")
print(f"h == 0:        {int((h_m == 0).sum()):,}    (must equal safe nodes)")
print(f"h: min {h_m.min()} m | median {int(np.median(h_m))} m | max {h_m.max()} m")

nodes:         6,749    (expect 6,750)
safe nodes:    1,038    (expect 1,039)
display nodes: 5,718    (expect 5,719)
h == 0:        1,038    (must equal safe nodes)
h: min 0 m | median 277 m | max 3861 m


## 4. Edge table, street names, and road classes
Each road segment becomes one record: which intersection it leaves, which it arrives at, how long it is, what kind of road, what it is called. Direction matters, so a two-way street is two records. One-ways are one, which is the point of storing direction at all.
Three cleanups happen here. Self-loops, a road leaving a corner and returning to it, get dropped, since you cannot route on them. Parallel segments, two roads between the same pair of corners, get reduced to the shorter one, since nobody would drive the longer. Both get counted rather than assumed.
Curve shapes only get stored when the road bends more than 10 meters off the straight line between its endpoints. Under that, the straight line is closer than the road is wide. Endpoints are left out of the stored shape, because they are already the two intersections.
Each segment also records the position of its reverse, so a hazard dropped on a street blocks both directions in one move.

In [6]:
from shapely.geometry import LineString, Point

GEOM_TOL_M = 10.0  # how far a road must bend before its shape is worth the bytes

edges      = []
seen       = {}   # (u,v) -> position in edges, catches parallel segments
street_ids = {}   # street name -> index into streets
class_ids  = {}   # road class -> index into classes
n_raw = n_self = n_dupe = n_geom = 0

def first_of(val):
    """OSM tags are sometimes a list of values. Take the first, consistently."""
    return (val[0] if val else None) if isinstance(val, list) else val

for u_osm, v_osm, data in G_halo.edges(data=True):
    n_raw += 1
    u, v = idx_of[u_osm], idx_of[v_osm]

    # A road from a corner back to the same corner. Real in OSM, useless for routing.
    if u == v:
        n_self += 1
        continue

    length = int(round(data.get('length', 0)))

    # Street name -> index. None stays None; plenty of segments are unnamed.
    nm = first_of(data.get('name'))
    if nm is None:
        name_i = None
    else:
        name_i = street_ids.setdefault(nm, len(street_ids))

    # Road class -> index. Feeds future hazard spread rules and weighting.
    cl = first_of(data.get('highway')) or 'unknown'
    cls_i = class_ids.setdefault(cl, len(class_ids))

    rec = {'u': u, 'v': v, 'len': length, 'cls': cls_i, 'name': name_i}

    # Curve shape, but only if the road actually curves.
    geom = data.get('geometry')
    if geom is not None and len(geom.coords) > 2:
        gx, gy = to_albers.transform(*geom.xy)          # meters, so 10 means 10
        straight = LineString([(X[u], Y[u]), (X[v], Y[v])])
        dev = max(straight.distance(Point(px, py))
                  for px, py in list(zip(gx, gy))[1:-1])   # interior vertices only
        if dev > GEOM_TOL_M:
            # Store interior vertices only. The ends are u and v, already known.
            rec['geom'] = [[round(lon, 5), round(lat, 5)]
                           for lon, lat in list(geom.coords)[1:-1]]
            n_geom += 1

    # Parallel segment: keep the shorter one, drop the other.
    if (u, v) in seen:
        n_dupe += 1
        j = seen[(u, v)]
        if length < edges[j]['len']:
            edges[j] = rec
        continue

    seen[(u, v)] = len(edges)
    edges.append(rec)

# Second pass: point each segment at its reverse, so one hazard blocks both ways.
pos = {(e['u'], e['v']): i for i, e in enumerate(edges)}
for e in edges:
    e['pair'] = pos.get((e['v'], e['u']))

streets = [s for s, _ in sorted(street_ids.items(), key=lambda kv: kv[1])]
classes = [c for c, _ in sorted(class_ids.items(), key=lambda kv: kv[1])]
n_oneway = sum(1 for e in edges if e['pair'] is None)

print(f"raw edges:     {n_raw:,}    (expect 16,597)")
print(f"self loops:    {n_self:,}    dropped")
print(f"parallel:      {n_dupe:,}    dropped")
print(f"final edges:   {len(edges):,}")
print(f"one-way:       {n_oneway:,}  ({n_oneway/len(edges):.1%})")
print(f"with geom:     {n_geom:,}  ({n_geom/len(edges):.1%})")
print(f"streets:       {len(streets):,}")
print(f"classes:       {len(classes)}  ->  {classes}")

raw edges:     16,595    (expect 16,597)
self loops:    135    dropped
parallel:      206    dropped
final edges:   16,254
one-way:       808  (5.0%)
with geom:     3,770  (23.2%)
streets:       2,826
classes:       11  ->  ['motorway', 'motorway_link', 'residential', 'tertiary', 'secondary', 'primary', 'unclassified', 'secondary_link', 'living_street', 'primary_link', 'tertiary_link']


## 5. Assemble, check, and measure
The three tables get written to one JSON file, compact, no whitespace. Before writing, three checks run: every segment points at intersections that exist, no intersection is stranded with no roads at all, and the reverse-lookup is symmetric.
Then the file gets gzipped in memory to measure it. Cloudflare compresses on delivery for free, so the gzipped number is the one that matters. The target is under 400KB. Over that, the curve tolerance is the first dial to turn.

In [7]:
import json, gzip, datetime

nodes = [{
    'x': round(float(lon[i]), 5),
    'y': round(float(lat[i]), 5),
    'safe': int(safe_mask[i]),
    'h': int(h_m[i]),
} for i in range(len(osm_ids))]

graph = {
    'meta': {
        'place': place,
        'halo_m': HALO_M,
        'geom_tol_m': GEOM_TOL_M,
        'built': datetime.date.today().isoformat(),
        'note': 'Computed wide, displayed narrow. Node index joins to id in egress_santa_rosa.geojson.',
    },
    'nodes': nodes,
    'edges': edges,
    'streets': streets,
    'classes': classes,
}

# --- Checks. Cheap now, expensive to discover in JavaScript later. ---
N = len(nodes)
bad_ref  = [e for e in edges if not (0 <= e['u'] < N and 0 <= e['v'] < N)]
touched  = {e['u'] for e in edges} | {e['v'] for e in edges}
stranded = N - len(touched)
# If A points at B as its reverse, B must point back at A.
bad_pair = [i for i, e in enumerate(edges)
            if e['pair'] is not None and edges[e['pair']]['pair'] != i]

print(f"bad node refs:   {len(bad_ref)}   (must be 0)")
print(f"no roads at all: {stranded}   (must be 0)")
print(f"broken pairs:    {len(bad_pair)}   (must be 0)")

raw = json.dumps(graph, separators=(',', ':')).encode('utf-8')
gz  = gzip.compress(raw, 9)

with open('graph_santa_rosa.json', 'wb') as f:
    f.write(raw)

print()
print(f"raw:      {len(raw)/1e6:.2f} MB")
print(f"gzipped:  {len(gz)/1e3:.0f} KB    (target under 400)")
print(f"ratio:    {len(raw)/len(gz):.1f}x")
print(f"per edge: {len(raw)/len(edges):.0f} bytes raw")

# Eyeball one curvy segment. Unreadable files hide bugs.
sample = next(e for e in edges if 'geom' in e)
print()
print("sample curved edge:", json.dumps(sample)[:220])

bad node refs:   0   (must be 0)
no roads at all: 0   (must be 0)
broken pairs:    0   (must be 0)

raw:      2.09 MB
gzipped:  433 KB    (target under 400)
ratio:    4.8x
per edge: 129 bytes raw

sample curved edge: {"u": 1, "v": 134, "len": 367, "cls": 1, "name": null, "geom": [[-122.72596, 38.44875], [-122.7258, 38.44824], [-122.72566, 38.44788], [-122.7255, 38.44753], [-122.72538, 38.44729], [-122.72527, 38.44708], [-122.72512, 3


## 6. Thin the curve vertices
OSM records road curves at survey fidelity, roughly a point every 30 meters. A map does not need that. Douglas-Peucker simplification drops any vertex sitting within 2 meters of the line between its neighbors, which is a quarter of a lane width and invisible on screen.
No curve gets deleted. Every one of the curved segments keeps its shape, described with fewer points. The simplification runs in projected meters, so 2 means 2.

In [9]:
SIMPLIFY_M = 2.0  # a quarter of a lane width. Invisible, and cheap to store.

to_wgs = Transformer.from_crs("EPSG:3310", "EPSG:4326", always_xy=True)
before = after = 0

for e in edges:
    if 'geom' not in e:
        continue

    # Full line including endpoints, so simplify can judge the first and last curves too.
    full_ll = [[nodes[e['u']]['x'], nodes[e['u']]['y']]] + e['geom'] + \
              [[nodes[e['v']]['x'], nodes[e['v']]['y']]]
    before += len(e['geom'])

    # Project -> simplify in meters -> project back.
    px, py = to_albers.transform(*zip(*full_ll))
    simp   = LineString(zip(px, py)).simplify(SIMPLIFY_M, preserve_topology=False)
    bx, by = to_wgs.transform(*zip(*simp.coords))

    # Strip the endpoints back off. They are u and v, already stored.
    interior = [[round(a, 5), round(b, 5)] for a, b in list(zip(bx, by))[1:-1]]

    if interior:
        e['geom'] = interior
        after += len(interior)
    else:
        # Simplified flat. It was never really a curve. Let it go.
        del e['geom']

still = sum(1 for e in edges if 'geom' in e)
graph['meta']['simplify_m'] = SIMPLIFY_M

raw2 = json.dumps(graph, separators=(',',':')).encode('utf-8')
gz2  = gzip.compress(raw2, 9)
with open('graph_santa_rosa.json', 'wb') as f:
    f.write(raw2)

print(f"vertices:  {before:,} -> {after:,}   ({1 - after/before:.0%} fewer)")
print(f"curves:    {n_geom:,} -> {still:,}   (flattened: {n_geom - still:,})")
print()
print(f"raw:       {len(raw2)/1e6:.2f} MB")
print(f"gzipped:   {len(gz2)/1e3:.0f} KB   (was 433, target under 400)")
print(f"floor:     249 KB with no curves at all")

vertices:  32,700 -> 14,774   (55% fewer)
curves:    3,770 -> 3,564   (flattened: 206)

raw:       1.70 MB
gzipped:   345 KB   (was 433, target under 400)
floor:     249 KB with no curves at all


## 7. Add join IDs to the published map file
The map file and the routing file need a shared key. We download the live GeoJSON from the repo rather than regenerating it, so the scores stay exactly as published and no 50 minute run is needed.
Each dot gets matched to its intersection by coordinate, rounded to five decimals, about one meter. Two checks guard this: no two intersections may collapse to the same rounded key, and every single dot must find a match. Anything less than 100% and the file does not get written, because a dot with no ID is a click that silently does nothing.

In [11]:
import urllib.request

URL = ("https://raw.githubusercontent.com/jerrod-lessel/egress-index/"
       "main/egress_santa_rosa.geojson")

with urllib.request.urlopen(URL) as r:
    gj = json.load(r)

feats = gj['features']
print(f"downloaded: {len(feats):,} features   (expect 5,718)")

# Key every intersection by its rounded position.
key_of = {}
collide = 0
for i in range(len(osm_ids)):
    k = (round(float(lon[i]), 5), round(float(lat[i]), 5))
    if k in key_of:
        collide += 1          # two intersections inside a meter of each other
    key_of[k] = i

print(f"key collisions: {collide}   (must be 0)")

hit = miss = 0
for f in feats:
    cx, cy = f['geometry']['coordinates'][:2]
    k = (round(float(cx), 5), round(float(cy), 5))
    if k in key_of:
        f['properties']['id'] = key_of[k]
        hit += 1
    else:
        miss += 1

print(f"matched: {hit:,}   unmatched: {miss:,}   (must be 0)")

if collide == 0 and miss == 0:
    with open('egress_santa_rosa.geojson', 'w') as fh:
        json.dump(gj, fh, separators=(',', ':'))
    ids = {f['properties']['id'] for f in feats}
    print(f"\nwritten. unique ids: {len(ids):,}  (must equal matched)")
    print(f"all ids point at real nodes: {max(ids) < len(nodes)}")
    print(f"sample: {json.dumps(feats[0]['properties'])}")
else:
    print("\nNOT WRITTEN. Fix the above first.")

downloaded: 5,718 features   (expect 5,718)
key collisions: 0   (must be 0)
matched: 5,718   unmatched: 0   (must be 0)

written. unique ids: 5,718  (must equal matched)
all ids point at real nodes: True
sample: {"safe": false, "score": 3, "pocket": null, "road": null, "id": 3142}


## 8. Download files

In [12]:
from google.colab import files
files.download('graph_santa_rosa.json')
files.download('egress_santa_rosa.geojson')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>